In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent.parent.resolve()))
from core.config import load_env_vars

load_env_vars()
from core.config import PROJECT_DIR

# Fitting the model (User does this)

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

import sys
from pathlib import Path
from core.config import DATA_DIR
import pickle
with open(DATA_DIR / 'boston_housing_dataset.pkl', 'rb') as rf:
    data_dict = pickle.load(rf)


data = data_dict['data']
metadata = data_dict['metadata']

# === Separate features and target ===
X = data.drop(columns=['MEDV'])
y = data['MEDV']

# === Train/test split ===
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# === Fit a baseline model ===
model = RandomForestRegressor(random_state=42, n_estimators=200)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(f"R²: {r2_score(y_test, y_pred):.3f}")
print(f"MAE: {mean_absolute_error(y_test, y_pred):.3f}")


R²: 0.884
MAE: 2.041


# The agent should be doing, this but for V1, this done manually.

In [3]:
from core.backend.xai import ceteris_paribus_bytes_multi
from core.utils import load_and_show_png_bytes

cp_bytes = ceteris_paribus_bytes_multi(model, X_test, num_datapoints=3, grid_points=100)

print(cp_bytes.keys())

dict_keys(['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX', 'PTRATIO', 'B', 'LSTAT'])


In [4]:
system_prompt_path = PROJECT_DIR / 'assets'/ 'SP_ethics_compliance.txt'
guidelines_path =  PROJECT_DIR / 'assets'/ 'guidelines' / 'guidelines_shorter.txt'
with open(system_prompt_path) as rf:
    system_prompt_template = rf.read()
with open(guidelines_path) as rf:
    guidelines = rf.read()
    

In [5]:
from core.backend.llms import make_text_generation_model_open_router
import base64
import pprint
from langchain_core.messages import HumanMessage, SystemMessage
from IPython.display import Image as im
from IPython.display import display as dis
from core.utils import describe_sklearn_model, describe_pandas_dataset, pretty_messages_pretty
from core.backend.llms import run_model
import asyncio
import pprint

# Initialize the language model with a specific model ID
llm = make_text_generation_model_open_router( # get vision language models here : https://openrouter.ai/models?fmt=cards&input_modalities=image&max_price=0
    # model_id='z-ai/glm-4.5-air:free'
    # model_id='google/gemini-2.0-flash-exp:free'
    model_id='nvidia/nemotron-nano-12b-v2-vl:free'
    # model_id='google/gemma-3-27b-it:free'
    # model_id='mistralai/mistral-small-3.1-24b-instruct:free'
)

image_bytes = cp_bytes['B'][0]
image_data = base64.b64encode(image_bytes).decode("utf-8")

model_info = describe_sklearn_model(model)
data_info = describe_pandas_dataset(data, metadata=metadata)

system_prompt_str = system_prompt_template.format(guidelines=guidelines)

system_prompt = SystemMessage(content=system_prompt_str)

user_prompt = HumanMessage(
    content=[
        {
            "type": "text",
            "text": "project details :The project is to do housing price prediction"
        },
        {
            "type": "text",
            "text": f'Data info : {pprint.pformat(metadata)}'
        },
        {
            "type": "image_url",
            "image_url": {"url": f"data:image/png;base64,{image_data}"},
            "image description": "explainability plot for a feature of the dataset"
        },
        {
            "type": "text",
            "text": "Given the project context, and the explainability results, Are any company guidelines violated?"
        },
    ]
)
    
response = await run_model(llm=llm, system_prompt=system_prompt, user_prompt=user_prompt)

getting answer ...

**Project Context Violations Identified:**  
1. **Diversity, Non-Discrimination, and Fairness**  
   - **Violated Guideline:** "Identify protected attributes and proxies; perform subgroup performance and bias analyses; avoid unjustified disparate impact."  
   - **Reason:** The feature `"B"` directly incorporates racial demographic data (African American descent), which is a protected attribute under fairness frameworks like the U.S. Fair Housing Act. While the feature is mathematically transformed (`1000*(Bk - 0.63)^2`), its derivation from race makes it a **proxy for a protected attribute**. There is no evidence of bias analysis (e.g., subgroup fairness checks) or mitigation steps for potential disparate impact on protected groups.  

2. **Privacy and Data Governance**  
   - **Violated Guideline:** "Collect and use data lawfully with minimization, anonymization/pseudonymization where appropriate."  
   - **Reason:** The feature `"B"` uses demographic data at the 

In [15]:
from core.frontend.nb_maker import create_nb_w_context_wo_code

response_content = f"""\
# AI generated Response

{response['full_message_content']}
___"""

create_nb_w_context_wo_code(response_content)

Created initial notebook: C:\Users\skuma\projects\Data_Seance\core\AI_generated\ethics_report.ipynb
